In [ ]:
"""
SSD (Single Shot MultiBox Detector) с бэкбоном MobileNet для одноканальных изображений.
Основано на: Liu et al. "SSD: Single Shot MultiBox Detector" (arXiv:1512.02325)

Архитектура:
    Вход: [B, 1, 300, 300] — грейскейл изображения
    Выход: [B, 8732, 4] — локализация (смещения якорей)
           [B, 8732, NUM_CLASSES] — уверенность (логиты классов)

Где 8732 = 19*19*6 + 10*10*6 + 5*5*6 + 3*3*6 + 2*2*6 + 1*1*6 — общее число якорных боксов.
"""

In [1]:
import torch
from model import SSDMobileNet
from train import SSDTrainer
from RailMarkDset import RailMarkDset, collate_fn, CVS
from inner_logic import Detector
from eval import evaluate_map
from torch.utils.data import DataLoader
from utils import visualize_detections, train_transform, load_checkpoin
import numpy as np
import config

In [2]:
# main

print("Загрузка датасета...")
fold1 = r'D:\Projects\Rail_marking\data_gen_for_YOLO\generated_dataset\train'
# fold1 = r'test_val'
dataset = RailMarkDset(image_folder=fold1, label_folder=fold1)
dataloader = DataLoader(
    dataset,
    batch_size=config.BATCH_SIZE,
    shuffle=True,
    collate_fn=collate_fn,
    num_workers=0,
    drop_last=True)

# Шаг 3: Инициализация модели
print("Инициализация SSD-MobileNet...")
model = SSDMobileNet(num_classes=config.NUM_CLASSES)

# Подсчёт параметров
total_params = sum(p.numel() for p in model.parameters())
print(f"Всего параметров: {total_params:,}")

# Шаг 4: Обучение
print("Начало обучения...")
trainer = SSDTrainer(model)

# =================================================================
# Загрузить веста и оптимайзер, если продолжение обучения
checkpoin_path = r'models\SSDmn_rmark\SSDmn_rmark_20260831_1816\35_SSDmn_rmark_20260831_1816_2.pt.tar'
load_checkpoin(torch.load(checkpoin_path),
               model=model, optimizer=trainer.optimizer)
# =================================================================

trainer.fit(dataloader, num_epochs=100)

# Шаг 5: Оценка на валидации
print("Оценка модели...")
fold2 = r'D:\Projects\Rail_marking\data_gen_for_YOLO\generated_dataset\val'
# fold2 = r'test_val'
val_dataset = RailMarkDset(image_folder=fold2, label_folder=fold2)
val_loader = DataLoader(val_dataset, batch_size=4, collate_fn=collate_fn)

map_score = evaluate_map(model, val_loader)
print(f"mAP@0.5: {map_score:.4f}")


Загрузка датасета...
Инициализация SSD-MobileNet...
Всего параметров: 11,012,400
Начало обучения...
loading checkpoint


100%|██████████| 250/250 [02:43<00:00,  1.53it/s]


Epoch [1/100] Loc Loss: 0.1018 | Conf Loss: 0.9463


100%|██████████| 250/250 [03:12<00:00,  1.30it/s]


Epoch [2/100] Loc Loss: 0.0670 | Conf Loss: 0.4534


...

100%|██████████| 250/250 [02:29<00:00,  1.67it/s]


Epoch [99/100] Loc Loss: 0.0226 | Conf Loss: 0.0129
EarlyStopping counter: 3 out of 5


100%|██████████| 250/250 [02:29<00:00,  1.67it/s]


Epoch [100/100] Loc Loss: 0.0224 | Conf Loss: 0.0124
=> save checkpoint
Оценка модели...


FileNotFoundError: [WinError 3] Системе не удается найти указанный путь: 'D:\\Projects\\Rail_marking\\data_gen_for_YOLO\\generated_dataset\\val'

In [ ]:
# Шаг 6: Визуализация одного примера
print("Визуализация детекций...")
model.eval()
detector = Detector()

with torch.no_grad():
    img, boxes, labels = val_dataset[5]
    img_batch = img.unsqueeze(0)
    if torch.cuda.is_available():
        img_batch = img_batch.cuda()

    loc, conf = model(img_batch)
    dets = detector.detect(loc, conf, conf_thresh=0.25)

    img_vis = img.squeeze().numpy()

    print(f"Ground truth boxes (norm): {boxes.tolist()}")
    print(f"Detections (norm): {dets[0].cpu().numpy()}")

    visualize_detections(img_vis, dets[0])

Визуализация детекций...


NameError: name 'val_dataset' is not defined

In [ ]:
import csv
import matplotlib.pyplot as plt

path = r'models\SSDmn_rmark\SSDmn_rmark_20260828_1804\training_log.csv'

epoch = []
train_loss = []
val_loss = []

with open(path, 'r', encoding='utf-8') as f:
    reader = csv.reader(f)
    next(reader, None)
    for row in reader:
        epoch.append(int(row[0]))
        train_loss.append(float(row[1]))
        val_loss.append(float(row[2]))

fig, ax = plt.subplots(1, 2, figsize=(15, 5))
ax[0].plot(train_loss)
ax[1].plot(val_loss)